# 에이전틱 시스템의 학습

* 비모수적 학습
    * 관련 모델의 파라미터를 바꾸지 않고도 자동으로 성능을 변경하고 개선하는 기법
* 모수적 학습
    * 파운데이션 모델의 파라미터를 명시적으로 학습하거나 파인튜닝하는 기법

## 비모수적 학습

### 비모수적 예시 학습

* 가장 단순한 기법이 예시 학습
* 에이전트가 작업을 수행하는 동안 품질에 대한 측정값을 제공하고 그 예시들을 사용해 향후 성능을 개선

* 에이전트는 새로운 문제를 해결하기 위해 과거 사례 데이터베이스에서 정보를 가져옴
* 성공적인 예제를 영속 저장소에 보관했다가 다시 검색해 프롬프트의 예제로 제공하면 다양한 작업에서 성능이 크게 향상됨
* 성공적인 예제가 늘어나면 유형별, 텍스트 기반 검색, 의미론적 검색 등을 사용해 가장 관련성이 높은 성공 사례를 검색하는 것이 좋음

-> 이 기법은 에이전틱 작업 전체에 적용할 수도 있고 작업의 하위 부분에 독립적으로 적용할 수도 있음

### 리플렉시온

* 에이전트에 언어 기반 간단한 자기 비판 습관을 부여함
* 각 시도가 실패한 뒤 에이전트가 무엇이 잘못되었고 다음 시도를 어떻게 개선할지에 대해 짧은 성찰을 작성하는 방식
    * 이후 메모리 버퍼에 저장

* 리플렉시온 루프
    1. 액션 시퀀스 수행
    2. 시도 내역 로그에 기록
    3. 성찰 생성
    4. 메모리 업데이트
    5. 다음 실행에 성찰 주입

* 모델의 가중치에는 손대지 않고 모델 자체를 코치로 사용하는 방식으로 매우 가벼움

* 리플렉시온 패턴 구현 예시

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-5")

reflections = []

def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}

reflexion_prompt = f"""어떤 환경에 놓인 뒤 특정 과제를 수행하라는 지시를 받았던 과거 경험의 기록이 주어질 것입니다.
당신은 그 과제를 완수하는 데 실패했습니다. 환경을 요약하려고 하지 말고, 과제를 완수하기 위해 시도했던 전략과 진행 경로에 대해 생각해 보십시오.
당신이 했어야 하지만 하지 못했던 구체적인 행동들을 참고하여, 그 실수를 보완할 수 있는 간결하고 새로운 실행 계획을 세우십시오.
예를 들어, A와 B는 시도했지만 C를 잊어버렸다면, 해당 환경에서 C를 달성하기 위해 어떤 행동을 했어야 하는지 환경에 특화된 행동들로 계획을 세우면 됩니다.
이 계획은 나중에 같은 과제를 다시 풀 때 필요하게 될 것입니다. "Plan"이라는 단어 뒤에 자신의 계획을 작성하십시오.

Instruction:
저는 유제품이 들어 있지 않고 사과 맛이 포함된 칩스 버라이어티 팩을 찾고 있으며, 가격은 30달러 미만이어야 합니다.

[Search]

Action: search[dairy free and apple variety pack of chips]
Observation:
[검색 결과로 돌아가기]
페이지 1 (총 결과: 50)
[다음 >]
[B07HRFJWP8]
Enjoy Life Foods Soft Baked Ovals, Breakfast Bars, Nut Free Bars, Soy Free, Dairy Free, Non GMO, Gluten Free, Vegan, Variety Pack, 4 Boxes (20 Bars Total)
$100.0
[B01KMHY5PG]
Enjoy Life Soft Baked Chewy Bars, Variety Pack, Nut Free Bars, Soy Free, Dairy Free, Gluten Free, 6 Boxes (30 Total Bars)
$21.49
[B008D2X8C4]
Enjoy Life Lentil Chips Variety Pack, Dairy Free Chips, Soy Free, Nut Free, Non GMO, Vegan, Gluten Free, 24 Bags (0.8 oz)
$100.0

Action: think[앞의 두 상품은 제가 원하는 것이 아닙니다. 세 번째 상품이 제가 찾는, 유제품이 들어 있지 않고 사과 맛이 포함된 칩스 버라이어티 팩입니다.]
Observation: 알겠습니다.

Action: click[B008D2X8C4]
Observation:
[검색 결과로 돌아가기]
[< 이전]
flavor name [dill and sour cream][garlic & parmesan][light sea salt][margherita pizza][thai chili lime][variety pack]
size [0.8 ounce (pack of 24)][4 ounce (pack of 12)]
가격(Price): $100.0
평점(Rating): N.A.
[Description]
[Features]
[Reviews]
[Buy Now]

Action: think[이 상품에는 'variety pack'과 '0.8 ounce (pack of 24)' 옵션이 있고, 구매하기에 좋아 보입니다.]
Observation: 알겠습니다.

Action: click[variety pack]
Observation: variety pack을 클릭하셨습니다.

Action: click[0.8 ounce (pack of 24)]
Observation: 0.8 ounce (pack of 24)를 클릭하셨습니다.

Action: click[Buy Now]

STATUS: FAIL

Plan:
"""

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("reflexion", call_model)
builder.add_edge(START, "reflexion")
graph = builder.compile()

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                reflexion_prompt
            )
        ]
    }
)
reflections.append(result)

# update_memory 함수 테스트 (선택사항)
trial_log_path = "ch07/data/trial_logs.txt"
env_configs = [
    {
        'is_success': False,
        'skip': False,
        'memory': []
    }
]
update_memory(tiral_log_path, env_configs)

### 경험 학습

* 경험 학습은 비모수적 학습을 더 확장
* 에이전트가 자신의 경험을 데이터베이스에 계속 모으는 것은 동일, 경험 전반에 걸쳐 인사이트를 집계해 미래의 정책을 개선하는 추가 단계를 적용
* 가치 있는 인사이트는 승격하고 덜 유용한 인사이트는 강등하며 인사이트를 수정하는 과정을 동적으로 수행
* 이 연구는 작업 간 학습 과정을 추가함으로써 리플렉시온을 확장

* 경험 학습 예시

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START
from langchain_core.messages import HumanMessage

# LLM 초기화
llm = ChatOpenAI(model="gpt-5")

# LLM 호출 함수
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}

class InsightAgent:
    def __init__(self):
        self.insights = []
        self.promoted_insights = []
        self.demoted_insights = []
        self.reflections = []

    def generate_insight(self, observation):
        # LLM을 사용하여 관찰에 기반한 인사이트를 생성
        messages = [HumanMessage(content=f"다음 관찰을 바탕으로 인사이트를 생성하세요: '{observation}'")]

        # 상태 그래프 생성
        builder = StateGraph(MessagesState)
        builder.add_node("generate_insight", call_model)
        builder.add_edge(START, "generate_insight")
        graph = builder.compile()

        # 메시지와 함께 그래프 호출
        result = graph.invoke({"messages": messages})

        # 생성된 인사이트 추출
        generated_insight = result["messages"][-1].content
        self.insights.append(generated_insight)
        print(f"생성된 인사이트: {generated_insight}")
        return generated_insight

* 생성된 인사이트는 다른 규칙들과의 상대적 중요도에 따라 정기적으로 재평가되고 조정됨
* 학습된 규칙은 경험에서 파생된 다른 규칙과 비교해 상대적 중요도에 따라 정기적으로 재평가되고 조정됨

In [ ]:
def promote_insight(self, insight):
    if insight in self.insights:
        self.insights.remove(insight)
        self.promoted_insights.append(insight)
        print(f"승격된 인사이트: {insight}")
    else:
        print(f"'{insight}'인사이트를 찾을 수 없습니다.")

def demote_insight(self, insight):
    if insight in self.promoted_insights:
        self.promoted_insights.remove(insight)
        self.demoted_insights.append(insight)
        print(f"강등된 인사이트: {insight}")
    else:
        print(f"'{insight}'인사이트를 찾을 수 없습니다.")

def edit_insight(self, old_insight, new_insight):
    # 모든 리스트에서 확인
    if old_insight in self.insights:
        index = self.insights.index(old_insight)
        self.insights[index] = new_insight
    elif old_insight in self.promoted_insights:
        index = self.promoted_insights.index(old_insight)
        self.promoted_insights[index] = new_insight
    elif old_insight in self.demoted_insights:
        index = self.demoted_insights.index(old_insight)
        self.demoted_insights[index] = new_insight
    else:
        print(f"'{old_insight}'인사이트를 찾을 수 없습니다.")
        return
    print(f"수정된 인사이트: '{old_insight}' -> '{new_insight}'")

def show_insights(self):
    print("\n현재 인사이트:")
    print(f"인사이트: {self.insights}")
    print(f"승격된 인사이트: {self.promoted_insights}")
    print(f"강등된 인사이트: {self.demoted_insights}")

def reflect(self, reflexion_prompt):
    # 성찰을 위한 상태 그래프 생성
    builder = StateGraph(MessagesState)
    builder.add_node("reflection", call_model)
    builder.add_edge(START, "reflection")
    graph = builder.compile()

    # 성찰 프롬프트와 함께 그래프 호출
    result = graph.invoke(
        {
            "messages": [
                HumanMessage(
                    content=reflexion_prompt
                )
            ]
        }
    )
    reflection = result["messages"][-1].content
    self.reflections.append(reflection)
    print(f"성찰: {reflection}")

In [ ]:
agent = InsightAgent()

# 시뮬레이션된 관찰 시퀀스와 KPI 타겟 달성 여부
reports = [
    ("웹사이트 트래픽이 15% 증가했지만, 바운스율이 40%에서 55%로 급격히 증가했습니다.", 
        False),
    ("이메일 열람률이 25%로 향상되었지만, 20% 목표를 초과했습니다.", True),
    ("장바구니 포기율이 60%에서 68%로 증가했지만, 50% 목표를 놓쳤습니다.", 
        False),
    ("평균 주문 가치가 8% 증가했지만, 5% 증가 목표를 놓쳤습니다.", True),
    ("신규 구독자 수가 5% 감소했지만, 10% 성장 목표를 놓쳤습니다.", 
        False),
]
# 1) 보고서 기간 동안 인사이트 생성 및 우선순위 지정
for text, hit_target in reports:
    insight = agent.generate_insight(text)
    if hit_target:
        agent.promote_insight(insight)
    else:
        agent.demote_insight(insight)
# 2) 승격된 인사이트 중 하나를 사람이 참여하는 편집으로 개선
if agent.promoted_insights:
    original = agent.promoted_insights[0]
    agent.edit_insight(original, f'개선된 인사이트: {original} 방문자 경험 개선을 위한 랜딩 페이지 UX 변경 조사')
# 3) 에이전트의 최종 인사이트 상태 표시
agent.show_insights()
# 4) 최상위 인사이트를 바탕으로 개선 계획 성찰
reflection_prompt = (
    "승격된 인사이트를 바탕으로, 다음 분기에 실행할 수 있는 하나의 고영향 실험을 제안하세요:" + f"\n{agent.promoted_insights}"
)
agent.reflect(reflection_prompt)  # 기존에 정의된 메서드를 직접 호출

* 학습할 샘플 수가 매우 많은 경우에는 파인튜닝을 고려하는 것이 타당할 수 있음

## 모수적 학습: 파인튜닝

* 평가용 데이터가 있으면 이를 활용해 시스템의 성능을 개선할 수 있음
* 충분한 수의 예제가 확보되면 에이전틱 시스템이 수행하는 작업에 대한 성능을 개선하기 위해 모델 파인튜닝을 고려할 만한 시점이 올 수 있음
* 파인튜닝은 사전학습된 모델의 파라미터를 소폭 조정해 새로운 작업이나 데이터셋에 적응시키는 일반적인 접근 방식

### 대형 파운데이션 모델 파인튜닝

* 대부분의 개발자는 대형 파운데이션 모델을 사용해 에이전틱 시스템을 구축하기 시작함
* 이런 모델을 파인튜닝한다는 것은 특정 작업이나 도메인에 맞게 파라미터를 표적 조정하는 것
* 이 과정으로 개발자는 모델이 가진 방대한 지식을 특수한 애플리케이션에 맞게 적응시켜 일반적인 능력은 유지하면서도 특정 작업에서의 관련성과 효율성을 높일 수 있음

* 파인튜닝을 고려할만한 경우
    * 도메인 특화가 중요한 경우
    * 일관된 톤과 형식이 중요한 경우
    * 도구 및 API 호출이 매우 정확해야 하는 경우
    * 충분한 고품질 데이터와 예산이 있는 경우
    * 재학습 주기를 감당할 수 있는 경우

* 파인튜닝을 미루는 것이 좋은 경우
    * 빠른 프로토타이핑 단계이거나 사용량이 적은 경우
    * 모델의 진화 속도가 투자 대비 너무 빠른 경우
    * 자원이 제약된 경우

* 즉, 파인튜닝은 성능 요구사항, 데이터 가용성, 운영 역량이 모두 맞아떨어지는 경우에만 수행해야 함
* 파인튜닝에 투자하기 전 먼저 기존 사전학습 또는 인스트럭션 튜닝된 모델이 프롬프트 엔지니어링, 비모수적 학습, 경량 적응 기법만으로 요구사항을 충족할 수 있는지 꼭 검토해야 함

* 언어 모델 파인 튜닝의 주요 방법

| 방법 | 작동 방식 | 적합한 용도 |
|-----|-----|-----|
| 지도 파인튜닝(SFT) | (프롬프트, 이상적인 응답) 쌍을 정답 예제로 제공<br>OpenAI 파인튜닝 API를 호출해 모델 가중치를 조정함 | 분류, 구조화된 출력, 지침 수행 실패 보정 |
| 비전 파인튜닝 | 이미지-레이블 쌍을 제공해 시각 입력에 대한 지도학습을 수행<br>이를 통해 이미지 이해와 멀티모달 지침 수행 능력을 향상시킴 | 이미지 분류, 멀티모달 지침 수행 안정성 |
| 직접 선호 최적화(DPO) | 각 프롬프트에 대해 좋은 응답과 나쁜 응답을 함께 제공하고 어느 쪽이 더 바람직한지 표시 모델은 더 높은 품질의 출력을 선호하도록 학습 | 요약 집중도 조정, 톤/스타일 제어 |
| 강화 파인튜닝(RFT) | 후보 출력들을 생성하고 전문가 평가자가 점수를 매김<br>이후 정책 경사 스타일 업데이트로 고득점 추론 과정을 강화 | 복잡한 추론, 도메인 특화 작업 |

* 대형 파운데이션 모델은 방대한 양의 일반 지식을 흡수하는 데 뛰어나지만 실제 힘은 도메인 특화 데이터로 파인튜닝했을 때 드러남
* 다만 대형 모델을 파인튜닝하려면 상당한 자원이 필요함
* 또한 중요한 요소가 고품질의 작업 특화 학습 데이터
    * 대형 모델이 특정 도메인에서 더 나아지려면 대표성 있는 예제를 충분히 봐야 함
    * 주의 깊게 처리하지 않으면 편향을 초래하고 과적합되어 일반화 능력과 공정성을 잃을 위험이 있음

### 소형 모델

* 대형 파운데이션 모델과 달리, 소형 모델은 더 적은 자원을 사용하는 대안으로 계산 제한되어 있거나 응답 시간이 중요한 많은 애플리케이션에 적합함
* 소형 모델은 파라미터 수가 적고 아키텍처가 더 단순하지만 특정 작업에 정교하게 파인튜닝되면 매우 효과적일 수 있음

* 소형 모델의 장점
    * 소형 모델의 경량 아키텍처는 투명성과 해석 가능성 측면에서 고유한 이점을 제공
    * 애자일 워크플로도 가능하게 함
    * 비용과 접근성 측면에서 용이함
    * 과적합 없이도 효과적으로 작동하도록 커스터마이징 가능
    * 잦은 업데이트나 재학습이 필요한 환경에서도 신속하게 재학습이나 파인튜닝되어 변화되는 패턴에 적응할 수 있음

* 소형 모델을 선택할 때는 지연시간, 하드웨어, 예산 같은 배포 제약과 작업 요구사항을 함께 고려해야 함

### SFT: 지도 파인튜닝

* 선별된 입력/출력 예제를 통해 행동을 정밀하게 조정할 수 있게 해주는 기초적인 기법
* 에이전트에 어떻게 응답해야 하는지를 명시적인 예제로 보여줌으로써 에이전트의 행동을 정밀하게 조정하는 기본 접근 방식

* 프롬프트 엔지니어링만으로는 부족할 때 더 높은 제어력과 일관성을 제공
* SFT는 주의 깊게 선별된 (프롬프트, 응답) 쌍을 사용해 모델이 원하는 출력 스타일, 구조, 행동을 학습하도록 돕는 기법

* SFT 워크플로
    1. 광범위한 코퍼스로 사전학습해 일반적인 능력을 쌓음
    2. 작업 특화 지도학습 데이터셋으로 추가 파인튜닝으로 특수화된 애플리케이션에 맞게 적응

* 함수 호출을 견고하게 만들기 위해 일반적으로 노출하는 각 API에 명시적인 스키마를 정의
    * 이 접근 방식의 경우 추가 데이터 큐레이션, 계산 자원, 유지 관리가 필요하기 때문에 사전학습 모델이 제공하는 기본 함수 호출 기능과 런타임 스키마 검증을 사용하는 것을 권장
    * 이 작업은 에이전트가 함수 호출 여부를 선택하고 인자를 정확히 채우고 결과를 적절히 래핑해야 하는 구조화된 예제를 모델에 제시하는 것을 포함

* 파인튜닝은 또한 모델이 사용자 입력을 유효한 인자로 파싱하고 누락된 파라미터 같은 오류에서 복구하며 함수 호출이 실패했을 때 우아하게 폴백하는 방법을 학습하도록 도움
    * 내부 추론을 감싸거나 호출을 넣는 것 같은 특수 토큰과 포맷팅을 사용하면 모델이 대화, 생각, 행동을 구분하기가 쉬워짐

* SFT로 파인튜닝하는 예시

In [ ]:
def build_preprocess_fn(tokenizer):
    """원시 샘플을 토크나이즈된 프롬프트로 매핑하는 함수를 반환합니다."""
    def _preprocess(sample):
        messages = sample["messages"].copy()
        _merge_system_into_first_user(messages)
        prompt = tokenizer.apply_chat_template(messages, tokenize=False)
        return {"text": prompt}

    return _preprocess

In [ ]:
def build_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        pad_token=ChatmlSpecialTokens.pad_token.value,
        additional_special_tokens=ChatmlSpecialTokens.list(),
    )
    tokenizer.chat_template = CHAT_TEMPLATE
    return tokenizer


def build_model(model_name: str, tokenizer, load_4bit: bool = False):
    """모델 로드. Mac에서는 bitsandbytes가 불안정하므로 4bit 양자화 비활성화."""
    kwargs = {
        "attn_implementation": "eager",  # flash_attn 호환성 (Phi-3 window_size 등)
        "device_map": "auto",
        "torch_dtype": torch.bfloat16,
    }
    # load_4bit=True이고 Mac이 아닐 때만 4bit 양자화 사용 (bitsandbytes는 Mac에서 불안정)
    use_4bit = load_4bit and platform.system() != "Darwin"
    if use_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.resize_token_embeddings(len(tokenizer))
    return model

In [ ]:
def load_and_prepare_dataset(ds_name: str, tokenizer, max_train: int, max_eval: int) -> DatasetDict:
    """데이터셋을 로드하고 전처리 및 학습/테스트 분할을 적용합니다."""
    raw = load_dataset(ds_name).rename_column("conversations", "messages")
    processed = raw.map(build_preprocess_fn(tokenizer), remove_columns="messages")
    split = processed["train"].train_test_split(test_size=0.1, seed=42)
    split["train"] = split["train"].select(range(max_train))
    split["test"] = split["test"].select(range(max_eval))
    return split

def train(
    model,
    tokenizer,
    dataset: DatasetDict,
    peft_cfg: LoraConfig,
    output_dir: str,
    epochs: int = 1,
    lr: float = 1e-4,
    batch_size: int = 1,
    grad_accum: int = 4,
    max_seq_len: int = 1500,
):
    train_args = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        save_strategy="no",
        eval_strategy="epoch",
        logging_steps=5,
        learning_rate=lr,
        num_train_epochs=epochs,
        max_grad_norm=1.0,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        report_to=None,
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        packing=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=train_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        processing_class=tokenizer,
        peft_config=peft_cfg,
    )

    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    return trainer

### DPO: 직접 선호 최적화

* 선호 학습을 도입해 출력이 사람의 품질 순위 판단에 더 가깝게 정렬되도록 함
* DPO는 순위가 매겨진 응답 쌍으로부터 학습해 더 나은 출력을 덜 좋은 출력보다 선호하도록 모델을 학습시키는 파인튜닝 기법

* DPO 워크플로
    1. 프롬프트를 모델에 넣어 여러 개의 완성을 생성
    2. 사람이 평가해 더 나은 응답에 대한 선호 데이터를 생성
    3. 직접 선호 최적화

* DPO를 사용하여 파인튜닝하는 예시

In [ ]:
import logging
import os
import platform
import torch
from datasets import load_dataset
from huggingface_hub import constants as hf_constants
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import DPOConfig, DPOTrainer

BASE_SFT_CKPT = "microsoft/Phi-3-mini-4k-instruct"
DPO_DATA      = "dpo_it_help_desk_training_data.jsonl"                   # -> 경로 또는 HF 데이터셋
OUTPUT_DIR    = "phi3-mini-helpdesk-dpo"

def _is_model_cached(repo_id: str) -> bool:
    """Hugging Face 모델이 로컬 캐시에 있는지 확인"""
    if os.path.exists(repo_id) and os.path.isdir(repo_id):
        return True  # 로컬 경로
    cache_folder = "models--" + repo_id.replace("/", "--")
    cache_path = os.path.join(hf_constants.HF_HUB_CACHE, cache_folder)
    return os.path.exists(cache_path)

# 1) 모델 + 토크나이저 로드
tok = AutoTokenizer.from_pretrained(BASE_SFT_CKPT, padding_side="right",
                                    trust_remote_code=True)

logger = logging.getLogger(__name__)
if not _is_model_cached(BASE_SFT_CKPT):
    logger.warning("로컬 경로를 찾을 수 없습니다. Hub에서 '%s'를 다운로드합니다.", BASE_SFT_CKPT)

# 2) 양자화 설정 (4비트)
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

# 3) 기본 모델 로드 (양자화 적용)
base = AutoModelForCausalLM.from_pretrained(
    BASE_SFT_CKPT,
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=bnb_config
)

# 4) LoRA 설정 및 모델 준비
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
   target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",
                    "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base, lora_cfg)
print("✅  Phi-3 모델 로드 완료:", model.config.hidden_size, "hidden dim")

# 5) 데이터셋 로드
ds = load_dataset("json", data_files=DPO_DATA, split="train")

# 6) DPO 학습 설정
dpo_args = DPOConfig(
    output_dir              = OUTPUT_DIR,
    per_device_train_batch_size  = 4,
    gradient_accumulation_steps  = 4,
    learning_rate           = 5e-6,
    num_train_epochs        = 3.0,
    bf16                    = True,
    logging_steps           = 10,
    save_strategy           = "epoch",
    report_to               = None,
    beta                    = 0.1,
    loss_type               = "sigmoid",
    label_smoothing         = 0.0,
    max_prompt_length       = 4096,
    max_completion_length   = 4096,
    max_length              = 8192,
    label_pad_token_id      = -100,  # 라벨에서 무시할 패딩 (표준값)
    truncation_mode         = "keep_end",
    generate_during_eval    = False,
    disable_dropout         = False,
    reference_free          = True,
    model_init_kwargs       = None,
    ref_model_init_kwargs   = None,
)

# 7) DPO 트레이너 초기회
trainer = DPOTrainer(
    model,
    ref_model=None,           # reference_free=True이므로 참조 모델 불필요
    args=dpo_args,
    train_dataset=ds,
    processing_class=tok,     # padding_side="right" 등 설정 반영
)

# 8) 학습 실행 및 모델 저장
trainer.train()
trainer.save_model()
tok.save_pretrained(OUTPUT_DIR)
print(f"✅  모델 저장 완료: {OUTPUT_DIR}")

### RLVR: 검증 가능 보상 강화 학습

* 선호 기반 파인튜닝을 바탕으로 RLVR은 명시적이고 측정 가능한 보상 함수에 대한 정책 최적화를 도입

* 자동화 지표, 규칙 기반 검증기, 외부 스코어링 모델, 인간 평가자 등 구축할 수 있는 어떤 평가자와도 연결해 해당 보상에 직접 최적화할 수 있게 함  
-> 검증 가능한 평가 신호를 정의할 수 있는 거의 모든 작업에 대해 확장 가능하고 표적화된 개선 가능

* RLVR은 선호 학습과 강화 학습을 결합해 모델이 가치 점수를 예측하고 그에 다라 최적화함으로써 관측된 순위 정보를 넘어 일반화할 수 있게 함
* 순위가 매겨진 선호 데이터가 있거나 출력을 평가할 신뢰할 수 있는 스코어링 함수를 구축할 수 있을 때 특히 효과적

* 장점
    * 어떤 측정 가능한 신호에 대해서도 최적화할 수 있는 유연성
    * 가치 예측을 통한 관측 예제 너머의 일반화 능력
    * 인간 평가가 가능한 작업에 잘 맞음